# Experiments with fitting wavelet coefficients marginal distributions 

In [ ]:
import torch
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats, ndimage
import math 

import sys
from pathlib import Path

root = Path().resolve()

sys.path.insert(0, str(root / '../codes'))
from sde_routines_condi import *
from sde_routines import *
from potentials_builder import *
from filters_bank import * 
from utils import *
from utils_entropy import *
from check_moments import *
from potentials import *
from filters import *
from mala import *
from ortho_wavelet import *

sys.path.insert(0, str(root / '../data'))
from data_loader import *

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print(root)

## Data preprocessing

In [ ]:
# Orthogonal wavelet
W = DefineWavelet('Db', m=3, device=device)

Data = load_turbulence_1d()
print(Data.shape) 

n1 = 1024
Data = split_periodize_reshape(Data, n1)

print(Data.shape) 

for j in range(3):
    Data = W.decompose(Data)[1]

x1 = normalize(Data[:1000]).to(device) # choose here if you want less data 
print('Data shape after preprocessing:', Data.shape)
print('x1 shape:', x1.shape) 

n1, channels, M = x1.shape
print(f'(B, C, T) = ({n1}, {channels}, {M}).')
J = 7 
Q = 3

### Verify data qualitatively

In [ ]:
%matplotlib inline 
plot_time_series_row(x1, 6)

## Interactively investigate wavelet coefficients histograms: lin/log scale 

In [ ]:
from ipywidgets import FloatSlider, IntSlider, ToggleButtons, interact

def analyze_wavelets(wt):
    """Creates an interactive widget to analyze wavelet coefficient histograms.

    Args:
        wt (torch.Tensor): Wavelet coefficients tensor of shape (B, J, T)
    """
    n_wavelets = wt.shape[1]

    # Pre-extract and flatten numpy arrays for speed during widget updates
    wavelet_data = []
    for i in range(n_wavelets):
        vals = wt[:, i, :].detach().cpu().flatten().numpy()
        wavelet_data.append(vals)

    # Define the interactive plotting function
    def update_plot(ch, x_min, x_max, percentile, scale):
        vals = wavelet_data[ch]

        # Calculate metrics based on current parameters
        total_counts = len(vals)
        counts_in_support = np.sum((vals >= x_min) & (vals <= x_max))
        pct_in_support = (counts_in_support / total_counts) * 100

        # Calculate percentile value (e.g., 95th percentile of absolute values or raw values)
        # Using absolute values is often helpful for symmetric wavelet coefficients
        pct_val_pos = np.percentile(vals, percentile)
        pct_val_neg = np.percentile(vals, 100 - percentile)

        # ------------------------------------------------------------------
        # Plotting
        # ------------------------------------------------------------------
        fig, ax = plt.subplots(figsize=(10, 5))

        # Plot histogram, log or linear depending on the `scale` toggle
        counts, bins, patches = ax.hist(
            vals, bins=100, density=True, log=(scale == "log"), alpha=0.6, color="skyblue"
        )

        # Highlight the selected support region
        ax.axvspan(x_min, x_max, color="green", alpha=0.15, label="Selected Support")

        # Vertical lines for percentiles
        ax.axvline(
            pct_val_pos,
            color="red",
            linestyle="--",
            linewidth=1.5,
            label=f"{percentile}th Pct ({pct_val_pos:.4f})",
        )
        ax.axvline(
            pct_val_neg,
            color="orange",
            linestyle="--",
            linewidth=1.5,
            label=f"{100-percentile}th Pct ({pct_val_neg:.4f})",
        )

        # Formatting
        ax.set_title(f"Analysis | Channel {ch}", fontsize=14)
        ax.set_xlabel("Coefficient Value")
        ax.set_ylabel(f"Density ({scale.capitalize()} Scale)")
        ax.legend(loc="upper right")
        ax.grid(True, which="both", linestyle=":", alpha=0.5)

        plt.show()

        # ------------------------------------------------------------------
        # Print Summary Stats
        # ------------------------------------------------------------------
        print(f"--- Channel {ch} Analysis ---")
        print(
            f"Support [{x_min:.3f}, {x_max:.3f}]: {counts_in_support:,} / {total_counts:,} samples ({pct_in_support:.2f}%)"
        )
        print(f"{percentile}th Percentile Value: {pct_val_pos:.6f}")
        print(f"{100-percentile}th Percentile Value: {pct_val_neg:.6f}")

    # Set up slider ranges based on global min/max of the data to keep it intuitive
    all_min = float(np.min([np.min(w) for w in wavelet_data]))
    all_max = float(np.max([np.max(w) for w in wavelet_data]))
    # Round slightly outward for clean slider steps
    step = (all_max - all_min) / 200

    # Trigger the interactive dashboard
    interact(
        update_plot,
        ch=IntSlider(min=0, max=n_wavelets - 1, step=1, value=6, description="Channel"),
        x_min=FloatSlider(
            min=all_min, max=all_max, step=step, value=all_min, description="X Min"
        ),
        x_max=FloatSlider(
            min=all_min, max=all_max, step=step, value=all_max, description="X Max"
        ),
        percentile=FloatSlider(
            min=95.0, max=100.0, step=0.005, value=95.0, description="Percentile"
        ),
        scale=ToggleButtons(options=["log", "linear"], value="log", description="Y Scale"),
    )


# ------------------------------------------------------------------
# To run it, just pass your `wt` tensor:
# ------------------------------------------------------------------
filters = return_Filters(M, J, 1, device=device)
wt = torch.fft.ifft(torch.fft.fft(x1) * filters).real  # (B, J, T)
analyze_wavelets(wt)

## Fitting wavelet coefficients histograms with Generalized Gaussian windows model  

### Q = 1 

In [ ]:
filters, filters_Phi = return_Filters(M, J, 1, device=device, include_phi=True)
model = Scalar_GGD_KRegion(filters)
model.fit_reference(x1)

In [ ]:
plot_kregion_fit_with_windows(x1, model, label="J=8, Q=1")

In [ ]:
# choose one plot for the paper 
plot_kregion_fit_paper(x1, model, channel=6, label="J=8, Q=1")

### Q = 3

In [ ]:
filtersQ3 = return_Filters(M, J, 3, device=device)
modelQ3 = Scalar_GGD_KRegion(filtersQ3)
modelQ3.fit_reference(x1)

In [ ]:
plot_kregion_fit_with_windows(x1, modelQ3, label="J=8, Q=3")

In [ ]:
# choose one plot for the paper 
plot_kregion_fit_paper(x1, modelQ3, channel = 16, label="J=8, Q=3")

## Derive analytical shape of fit, compare generated and original data 

In [ ]:
for m, label in [(model, "Q=1"), (modelQ3, "Q=3")]:
    n_ch = m.filters.shape[1]
    ncols = 4
    nrows = math.ceil(n_ch / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes = axes.flatten()
    for i in range(n_ch):
        m.compare_channel(x1, j=i, ax=axes[i])
    for j in range(n_ch, len(axes)):
        axes[j].axis("off")
    plt.suptitle(f"{label}: fit vs generated vs analytic shape", fontsize=20)
    plt.tight_layout()
    plt.show()

## Compare real and imaginary part of wavelet coefficient plots 

In [ ]:
def plot_wavelet_real_imag(x1: torch.Tensor, filters: torch.Tensor,
                      label: str = "Wavelet",
                     fit_coshgt: bool = True) -> None:
    
    # 1. Ensure x1 has a singleton dimension for broadcasting over filters
    # If x1 is (B, T), unsqueeze(1) makes it (B, 1, T) so it broadcasts over (1, J, T) filters
    x1_fft = torch.fft.fft(x1.unsqueeze(1) if x1.ndim == 2 else x1)
    
    # 2. Perform the frequency-domain multiplication and IFFT
    wt_complex = torch.fft.ifft(x1_fft * filters)
    wt_imag = wt_complex.imag  # (B, J, T)
    wt_real = wt_complex.real  # (B, J, T)
    
    n_wavelets = filters.shape[1]
    
    # ------------------------------------------------------------------
    # 1. Overview grid
    # ------------------------------------------------------------------
    ncols = 5
    nrows = math.ceil(n_wavelets / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    
    # Handle edge case where n_wavelets = 1 (axes wouldn't be an array)
    if n_wavelets == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for i in range(n_wavelets):
        # .numpy() works smoothly if detached and moved to CPU
        vals_imag = wt_imag[:, i, :].detach().cpu().numpy().flatten()
        vals_real = wt_real[:, i, :].detach().cpu().numpy().flatten()
    
        axes[i].hist(vals_imag, bins=50, density=True, log=True, alpha=0.5, label="Imag")
        axes[i].hist(vals_real, bins=50, density=True, log=True, alpha=0.5, label="Real")
        axes[i].set_title(f"{label} ch={i}")
        axes[i].set_xlabel("Coefficient value")
        axes[i].set_ylabel("Density")
        if i == 0:
            axes[i].legend()  # Add a legend to the first plot for clarity
    
    # Hide unused subplots
    for j in range(n_wavelets, len(axes)):
        axes[j].axis("off")
    
    plt.suptitle(f"Wavelet coefficient histograms — {label}", fontsize=13)
    plt.tight_layout()
    
    # 3. If this code is inside a loop, use plt.draw() or plt.pause() instead, 
    # or clear the figure afterward so the next loop can plot.
    plt.show()

In [ ]:
filters = return_Filters(M, J, 1, device=device)
plot_wavelet_real_imag(x1, filters, label="Q=1") 
filters_Q = return_Filters(M, J, 3, device=device)
plot_wavelet_real_imag(x1, filters_Q, label="Q=3") 
